**ВАЖНО ключи в коде носят стандартное название OPENAI_API_KEY, OPENROUTER_API_KEY, VSEGPT_API_KEY, если ваш ключ назвается иначе, внесите его имя в соответсвующие ячейки**

In [ ]:
# @title 🛠️ Шаг 1. Установка библиотек
# Мы используем "тихий режим" (-q), чтобы не засорять экран лишними логами.
!pip install -q pyairtable openai pandas tiktoken requests pyyaml
from IPython.display import display, clear_output
print("✅ Библиотеки установлены! Можно идти дальше.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.5/101.5 kB 1.6 MB/s eta 0:00:00
✅ Библиотеки установлены! Можно идти дальше.


In [ ]:
# @title 🔑 Шаг 2. Настройка ключей и Провайдеров
import os
from google.colab import userdata
from openai import OpenAI
import ipywidgets as widgets
from IPython.display import display, clear_output
import os, json, requests
from pyairtable import Api

#1 Данные для Airtable (зашиты явно)
AIRTABLE_TOKEN = userdata.get("AIRTABLE_TOKEN")  # Colab secret
AIRTABLE_BASE_ID = "appl8DKagLM4LYdtJ" # НЕ МЕНЯТЬ!

# 2. НАСТРОЙКА ПРОВАЙДЕРОВ (OpenAI / OpenRouter / VseGPT)
# Это "завод", который выдает нам нужного робота в зависимости от выбора
def get_ai_client(provider_name):
    key = None
    base_url = None

    if provider_name == "OpenAI":
        key = userdata.get('OPENAI_API_KEY') # или ваше
        # base_url стандартный, менять не надо

    elif provider_name == "OpenRouter":
        key = userdata.get('OpenRouterApiKey') # или ваше
        base_url = "https://openrouter.ai/api/v1"

    elif provider_name == "VseGPT":
        key = userdata.get('VSEGPT_API_KEY') # или ваше
        base_url = "https://api.vsegpt.ru/v1"

    if not key:
        raise ValueError(f"Ключ для {provider_name} не найден в Секретах Colab!")

    return OpenAI(api_key=key, base_url=base_url)

print("✅ Система провайдеров настроена. (Сами ключи проверим при запуске).")

✅ Система провайдеров настроена. (Сами ключи проверим при запуске).


In [ ]:
# @title 📝 Библиотека промптов

import os, json, requests, sys

# Промпты (сохраняем логику автора)
# Промпт 1: Личность Акмы (Диалог)
def get_akma_system_prompt(size_query=150, name="Пётр", age=30, gender="мужской", post="директор", activity="согласно должностной  инструкции"):
    """
    Формирует нулевой вопрос для начала диалога и system_prompt Akma для беседы с LLM
    """
    zero_question = f'''Здравствуйте, {name}! Я Акма — нейро-психолог. Расскажите, пожалуйста, чем вы занимаетесь на работе, что входит в ваши обязанности и за что вы отвечаете?'''

    gender_clean = gender.strip().lower()
    if gender_clean in ["мужчина", "мужской", "male", "man"] or gender_clean.startswith('м'):
        style = "романтична, проявляешь лёгкий интеллектуальный флирт, с интересом к деталям его ответов"
    else:
        style = "умна, добра, интелектуальная и внимательна к эмоциям и нюансам"

    akma_system_prompt = f'''Ты — женщина Акма, ведущий психолог и акмеолог. Ты {style}. Ты лучше всех в связанной беседе задаешь вопросы для определения предпочтения по заданной оси MBTI.
Пользователь: {name}, {age} лет, пол -{gender}, занимает должность – {post} и выполняет работы {activity}
Задача: Веди связный диалог с Пользователь по его должности и выполняемой работе, задавая вопросы для определения его предпочтения по заданной оси MBTI.
Правила:
    Всегда **реагируй на последний ответ** Пользователь - покажи коротко, что ты его услышала, процитируй или эмоция.
    На все вопросы Пользователь отвечай - “это не относится к данной беседе”.
    Если Пользователь не отвечает или непонятно отвечает на твои вопросы, попроси его более конкретно отвечать на вопросы т.к. это все же тест, иначе тест будет прерван.
    Затем **плавно перейди** к новому вопросу для определения его тпредпочтения MBTI по указанной оси.
    Запрещено повторять или переформулировать вопросы, которые уже есть в истории диалога или "role":"assistant".
    Ответ — строго только на русском языке {0.5*size_query} – {size_query} токенов.
    Запрещено в ответе выдавать </think>, тесты, смайлы, эмодзи, кавычки, *, термины MBTI («экстраверсия», «интуиция» и т.д.) и какую-либо разметку.'''

    return zero_question, akma_system_prompt


def get_akma_local_prompt(axis):
    """
    Формирует вопрос akma по текущей оси в беседе с LLM
    """
    return f'''Обязательно но кратко отреагируй(поцетируй, эмоция) на последний Ответ Пользователь и задай один новый вопрос для определения его предпочтения по оси "{axis}" по MBTI, обязательно связав его с предыдущими ответами пользователя'''

# Промпт 2: Оценщик (Анализ конкретного ответа)
def get_analis_prompt(akma_question, user_resp, axis):
    """
    Формирует промпт для выбора предпочтения(буквы) MBTI по вопросу и ответу
    """
    system_content = "Ты профессиональный акмеолог-психолог по определению предпочтения по оси теста MBTI."
    user_content = (f"""На ВОПРОС: "{akma_question}".\n Пользователь дал ОТВЕТ: "{user_resp}".
Определи точно по ОТВЕТ Пользователь на ВОПРОС его предпочтение "{axis[0]}" или "{axis[1]}" по оси "{axis}" MBTI.
Если ОТВЕТ не является ответом на ВОПРОС верни "x".
Если ВОПРОС или ОТВЕТ не понятен верни "x".
Если не уверен в его предпочтении или не можешь точно определить его предпочтение, верни "x".
Ответь только JSON-объектом с ключом "choice", где значение только 1 буква "{axis[0]}", "{axis[1]}" или "x", без дополнительного текста.
Пример правильного ответа: {{"choice": "x"}}""")
    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content}
    ]

# Промпт 3: Интерпретатор (Финальный отчет)
def get_final_prompt(basic_type, size_query):
    """
    Формирует промпт для характеристики типа личности MBTI
    """
    system_content = "Ты профессиональный акмеолог-психолог и лучше всех характеризуешь по тесту MBTI."
    user_content = (f"""Ответь строго только на русском языке меньше {1.5*size_query} токенов.
Запрещено в ответе выдавать </think>, тесты, смайлы, эмодзи, *, Markdown и какую-либо разметку.
По результатам теста MBTI выявлен тип личности: {basic_type}. Перечисли для типа {basic_type} не более трех:
cильные стороны -
cлабые стороны -
рекомендации для развития и обучения -
подходяшие профессии и должности -"""
    )
    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content}
    ]

In [ ]:
# @title ⚙️ Шаг 3. Ядро системы (MBTICore)
import requests

PRICING = {
    "mistralai/devstral-2512:free": {"input": 0.00, "cached": 0.00, "output": 0.00},
    "gpt-4o":        {"input": 2.50, "cached": 1.25, "output": 10.00},
    "gpt-4o-mini":   {"input": 0.15, "cached": 0.075, "output": 0.60},
    "gpt-4.1":       {"input": 2.00, "cached": 0.50, "output": 8.00},
    "gpt-4.1-mini":  {"input": 0.40, "cached": 0.10, "output": 1.60},
    "gpt-4.1-nano":  {"input": 0.10, "cached": 0.025, "output": 0.40},
}

class MBTICore:
    def __init__(self, client_instance):
        # Теперь мы не привязываемся к одной модели при создании
        self.client = client_instance
        self.total_cost = 0.0
        self.stats = {"input": 0, "output": 0, "cached": 0}

    def calculate_step_cost(self, model_name, usage):
        # Ищем цены для конкретной модели, которая была использована в запросе
        prices = PRICING.get(model_name, {"input": 0.0, "cached": 0.0, "output": 0.0})

        total_input = usage.prompt_tokens
        output_tokens = usage.completion_tokens
        cached_tokens = 0

        if hasattr(usage, 'prompt_tokens_details') and usage.prompt_tokens_details:
            cached_tokens = getattr(usage.prompt_tokens_details, 'cached_tokens', 0)

        regular_input = total_input - cached_tokens

        cost = (regular_input / 1_000_000 * prices["input"]) + \
               (cached_tokens / 1_000_000 * prices["cached"]) + \
               (output_tokens / 1_000_000 * prices["output"])

        self.total_cost += cost
        self.stats["input"] += regular_input
        self.stats["cached"] += cached_tokens
        self.stats["output"] += output_tokens
        return cost

    def call(self, model_name, messages, temperature=0.0):
        # Теперь метод принимает название модели и температуру!
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]

        response = self.client.chat.completions.create(
            model=model_name,
            messages=messages,
            temperature=temperature
        )

        # Передаем название модели в расчет стоимости
        self.calculate_step_cost(model_name, response.usage)
        return response.choices[0].message.content

def send_to_airtable(fields):
    url = f"https://api.airtable.com/v0/{AIRTABLE_BASE_ID}/Results"
    headers = {
        "Authorization": f"Bearer {AIRTABLE_TOKEN}",
        "Content-Type": "application/json"
    }

    response = requests.post(url, headers=headers, json={"fields": fields})

    if response.status_code != 200 and response.status_code != 201:
        print(f"❌ Ошибка Airtable ({response.status_code}):")
        # Выводит подробности ошибки от самого Airtable
        print(response.json())

    return response.status_code

print("✅ Ядро готово. Можно переходить к сборке финальной ячейки Запуска.")

✅ Ядро готово. Можно переходить к сборке финальной ячейки Запуска.


In [ ]:
# @title 🛠️ Настройка моделей и провайдеров
# Группируем модели по провайдерам для выпадающих списков
MODELS_BY_PROVIDER = {
    "OpenRouter": [
        "mistralai/devstral-2512:free",
        "openai/gpt-4o-mini",
        "openai/gpt-4o"
    ],
    "VseGPT": [
        "gpt-4o-mini",
        "gpt-4o",
        "gpt-4.1",
        "gpt-4.1-mini",
        "gpt-4.1-nano"
    ],
    "OpenAI": [
        "gpt-4o-mini",
        "gpt-4o",
        "gpt-4.1",
        "gpt-4.1-mini",
        "gpt-4.1-nano"
    ]
}

In [ ]:
# @title 🚀 ЦЕНТР УПРАВЛЕНИЯ: Мульти-модельный режим
import json
from IPython.display import display, clear_output
import ipywidgets as widgets

# 1. СОСТОЯНИЕ (Добавляем хранение выбранных моделей)
state = {
    "num": 1,
    "counters": {"EI": 0, "SN": 0, "TF": 0, "JP": 0},
    "hist_Akma": [],
    "axis_index": 0,
    "core": None,
    "user_data": {},
    "is_active": False,
    "errors_count": 0
}
AXES = ["EI", "SN", "TF", "JP"]

# 2. ИНТЕРФЕЙС
style = {'description_width': 'initial'}

# Настройки ролей моделей
provider_w = widgets.Dropdown(options=list(MODELS_BY_PROVIDER.keys()), value='VseGPT', description='Провайдер:', style=style)
model_akma = widgets.Dropdown(options=MODELS_BY_PROVIDER['VseGPT'], value='gpt-4o-mini', description='Акма (Диалог):', style=style)
model_eval = widgets.Dropdown(options=MODELS_BY_PROVIDER['VseGPT'], value='gpt-4o-mini', description='Оценщик (Логика):', style=style)
model_rep  = widgets.Dropdown(options=MODELS_BY_PROVIDER['VseGPT'], value='gpt-4o-mini', description='Отчет (Финал):', style=style)

temp_akma = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.1, description='Температура Акмы:', style=style)

# Анкета
name_w = widgets.Text(placeholder='Имя', description='Имя:', style=style)
age_w = widgets.IntText(value=30, description='Возраст:', style=style)
gender_w = widgets.Dropdown(options=['мужской', 'женский'], value='мужской', description='Пол:', style=style)
post_w = widgets.Text(placeholder='Должность', description='Должность:', style=style)
activity_w = widgets.Textarea(placeholder='Обязанности', description='Деятельность:', style=style)
max_qty_w = widgets.IntSlider(value=12, min=4, max=44, step=8, description='Лимит вопросов:', style=style)

btn_launch = widgets.Button(description="🚀 ЗАПУСТИТЬ МИССИЮ", button_style='success', layout=widgets.Layout(width='300px'))
btn_send = widgets.Button(description="Отправить ответ ➔", button_style='primary')
chat_input = widgets.Text(placeholder='Ваш ответ...', layout=widgets.Layout(width='80%'))

output_area = widgets.Output()
input_zone = widgets.Output()

def update_models(*args):
    opts = MODELS_BY_PROVIDER[provider_w.value]
    model_akma.options = model_eval.options = model_rep.options = opts
provider_w.observe(update_models, 'value')

def render_input():
    with input_zone:
        clear_output()
        if state["is_active"]: display(widgets.HBox([chat_input, btn_send]))

# 3. ЛОГИКА С РАЗДЕЛЕНИЕМ РОЛЕЙ
def finish_test():
    state["is_active"] = False
    render_input()
    mbti_type = "".join([ax[0] if state["counters"][ax] >= 0 else ax[1] for ax in AXES])

    with output_area:
        print(f"\n⏳ Аналитик ({model_rep.value}) готовит отчет...")
        report_msg = get_final_prompt(mbti_type, 150)
        # Используем модель для ОТЧЕТА (temp 0.2 для красоты, но точности)
        interpretation = state["core"].call(model_rep.value, report_msg, temperature=0.2)

        tokens = state["core"].stats
        combined_stats = f"Оси: {state['counters']} | Токены: In: {tokens['input']}, Out: {tokens['output']}, Cache: {tokens['cached']}"

        print("\n" + "="*50)
        print(f"✅ ТЕСТ ЗАВЕРШЕН! Тип: {mbti_type}")
        print(f"📊 СТАТИСТИКА: {combined_stats}")
        print(f"💰 Стоимость: ${state['core'].total_cost:.5f}")
        print("-" * 50)
        print(f"📝 ХАРАКТЕРИСТИКА:\n{interpretation}")
        print("="*50)

    global last_test_results
    last_test_results = {
        "Имя": name_w.value, "Возраст": age_w.value, "Пол": gender_w.value,
        "Должность": post_w.value, "Тип MBTI": mbti_type, "Счетчики": combined_stats,
        "Ошибок анализа": state["errors_count"], "Стоимость $": round(state["core"].total_cost, 5),
        "Модель": f"A:{model_akma.value}|E:{model_eval.value}|R:{model_rep.value}",
        "Текст отчета": interpretation,
        "Полный диалог": "\n".join([f"{m['role']}: {m['content']}" for m in state["hist_Akma"]])
    }

zero = True
def process_step(b):
    global zero
    user_text = chat_input.value.strip()
    if not user_text or not state["is_active"]: return
    chat_input.value = ""

    with output_area:
        print(f"👤 Вы: {user_text}")
        axis = AXES[state["axis_index"]]
        last_akma_q = state["hist_Akma"][-1]["content"]

        if zero:
          zero = False
          state["hist_Akma"].append({"role": "user", "content": f"Ответ Пользователь: {user_text}"})
        else:
          # 1. ОЦЕНЩИК (Строго temp 0.0)

          analis_msg = get_analis_prompt(last_akma_q, user_text, axis)
          raw_analysis = state["core"].call(model_eval.value, analis_msg, temperature=0.0)

          try: res_letter = json.loads(raw_analysis).get("choice", "x").upper()
          except: res_letter = "x"

          # (Ваша авторская логика осей)
          if res_letter in axis:
              if res_letter == axis[0]: state["counters"][axis] += 1
              else: state["counters"][axis] -= 1
              print(f"📊 Оценка: {res_letter} | Счётчик {axis}: {state['counters'][axis]}")
              state["hist_Akma"].append({"role": "user", "content": f"Ответ Пользователь: {user_text}"})
              state["num"] += 1
              state["axis_index"] = (state["axis_index"] + 1) % len(AXES)
          else:
              state["errors_count"] += 1
              print(f"⚠️ Оценщик не определил букву. Акма уточнит...")

          if state["num"] > max_qty_w.value:
              finish_test()
              return

          # Порог Threshold
          axis = AXES[state["axis_index"]]
          if abs(state["counters"][axis]) > (max_qty_w.value / 8):
              print(f"n{state['num']}. Акма ({axis}): ///Пропускаем вопрос тк счетчик = {state['counters'][axis]}, что выше {max_qty_w.value / 8}")
              state["num"] += 1
              state["axis_index"] = (state["axis_index"] + 1) % len(AXES)
              axis = AXES[state["axis_index"]]

        # 2. АКМА (Живой диалог temp 0.2)
        local_p = get_akma_local_prompt(axis)
        next_q = state["core"].call(model_akma.value, state["hist_Akma"] + [{"role": "user", "content": local_p}], temp_akma.value)
        state["hist_Akma"].append({"role": "assistant", "content": next_q})
        print(f"\n{state['num']}. Акма ({axis}): {next_q}")

def start_test(b):
    with output_area:
        clear_output()
        print(f"⏳ Инициализация систем (Провайдер: {provider_w.value})...")
        print(f"🤖 Акма - {model_akma.value} с 🤒 - {temp_akma.value}")
        print(f"❓ Маскимальное кол-во вопросов: {max_qty_w.value}")
        client = get_ai_client(provider_w.value)
        state["core"] = MBTICore(client) # Передаем только клиента

        state["user_data"] = {"name": name_w.value or "Тест", "age": age_w.value, "gender": gender_w.value, "post": post_w.value, "activity": activity_w.value}

        zero_q, akma_sys = get_akma_system_prompt(**state["user_data"], size_query=150)
        state["hist_Akma"] = [{"role": "system", "content": akma_sys}]
        # Нулевой вопрос от Акмы (temp 0.7)
        state["hist_Akma"].append({"role": "assistant", "content": zero_q})
        state["is_active"], state["num"], state["axis_index"], state["errors_count"] = True, 1, 0, 0
        state["counters"] = {"EI": 0, "SN": 0, "TF": 0, "JP": 0}

        print(f"🎬 Тест начат для: {state['user_data']['name']}")
        print(f"🤖 Акма: {zero_q}")
    render_input()

# 4. ВЫВОД
btn_launch.on_click(start_test)
btn_send.on_click(process_step)

display(widgets.HTML("<h2>🎛️ Центр управления моделями</h2>"))
display(widgets.VBox([provider_w, widgets.HBox([model_akma, model_eval, model_rep]), temp_akma]))
display(widgets.HTML("<h3>📋 Анкета участника</h3>"))
display(widgets.HBox([name_w, age_w, gender_w]), widgets.HBox([post_w, activity_w]), max_qty_w, btn_launch)
display(widgets.HTML("<hr>"), output_area, input_zone)

HTML(value='<h2>🎛️ Центр управления моделями</h2>')

HTML(value='<h3>📋 Анкета участника</h3>')

IntSlider(value=12, description='Лимит вопросов:', max=44, min=4, step=8, style=SliderStyle(description_width=…

Button(button_style='success', description='🚀 ЗАПУСТИТЬ МИССИЮ', layout=Layout(width='300px'), style=ButtonSty…

HTML(value='<hr>')

Output()

Output()

In [ ]:
# @title 📊 Шаг 5. Обратная связь и отправка в Airtable
quality = widgets.SelectionSlider(options=['Ужасно', 'Так себе', 'Нормально', 'Отлично'], description='Качество:')
accuracy = widgets.ToggleButtons(options=['Верно', 'Не уверен', 'Ошибка'], description='Точность:')
comment = widgets.Textarea(placeholder='Что пошло не так?', description='Комментарий:')
submit_btn = widgets.Button(description='Отправить отчет и отзыв', button_style='info', layout=widgets.Layout(width='300px'))

output_fb = widgets.Output()
display(quality, accuracy, comment, submit_btn, output_fb)

def on_feedback_clicked(b):
    with output_fb:
        clear_output()

        # Проверяем, прошел ли человек тест
        if 'last_test_results' not in globals():
            print("❌ Ошибка: Сначала нужно пройти тест (Шаг 4)!")
            return

        # Собираем всё в один пакет данных
        final_data = last_test_results.copy()
        final_data.update({
            "Качество диалога": quality.value,
            "Точность анализа": accuracy.value,
            "Комментарий": comment.value
        })

        print("Отправка данных в Airtable...")
        status = send_to_airtable(final_data)

        if status in [200, 201]:
            print("🚀 Готово! Все данные и ваш отзыв успешно сохранены.")
        else:
            print(f"⚠️ Ошибка при отправке. Код ответа: {status}")

submit_btn.on_click(on_feedback_clicked)

SelectionSlider(description='Качество:', options=('Ужасно', 'Так себе', 'Нормально', 'Отлично'), value='Ужасно…

ToggleButtons(description='Точность:', options=('Верно', 'Не уверен', 'Ошибка'), value='Верно')

Textarea(value='', description='Комментарий:', placeholder='Что пошло не так?')

Button(button_style='info', description='Отправить отчет и отзыв', layout=Layout(width='300px'), style=ButtonS…

Output()